# Synthetic Title Catalog Ingestion
This notebook contains the code samples on ingesting the movie title catalog data into an index for a Amazon Opensearch Serverless collection.  

In [ ]:
# Make sure you download the latest botocore and boto3 libraries.
import shutil
import subprocess
import sys

def ensure_uv_installed():
    if shutil.which("uv") is None:
        print("🔧 'uv' not found. Installing with pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"])
    else:
        print("✅ 'uv' is already installed.")

def uv_install(*packages):
    ensure_uv_installed()
    uv_path = shutil.which("uv")
    print(f"📦 Installing {', '.join(packages)} using uv...")
    subprocess.check_call([uv_path, "pip", "install", *packages])

uv_install("botocore", "boto3")

install dependencies

In [ ]:
!uv pip install -r requirements.txt

In [ ]:
import json
import boto3
import os
import logging

Provide the AWS credentials to access the Opensearch Serverless Collection

In [ ]:
os.environ["AWS_PROFILE"] = "your AWS Profile" # or use your AWS Access Keys
os.environ["AWS_DEFAULT_REGION"] = "your region"

In [ ]:
with open("../../data/movie_dataset.json", "r") as f:
    data = f.read()

In [ ]:
data_dict = json.loads(data)

In [ ]:
bedrock_runtime_client = boto3.client('bedrock-runtime')
embedding_model_id = "amazon.nova-2-multimodal-embeddings-v1:0"
embedding_dim = 1024

logger = logging.getLogger(__name__)

# Creates embeddings for the title information

In [ ]:
from typing import Any, Dict, Generator, List, Optional

def invoke_model(input_body: Dict[str, Any] = {}) -> Dict[str, Any]:
        body = json.dumps(input_body)

        try:
            response = bedrock_runtime_client.invoke_model(
                body=body,
                modelId=embedding_model_id,
                accept="application/json",
                contentType="application/json",
            )

            response_metadata = response.get("ResponseMetadata", {})
            if response_metadata:
                logger.info(
                    f"Successfully invoked model {embedding_model_id}. "
                    f"ResponseMetadata: {response_metadata}"
                )

            response_body = json.loads(response.get("body").read())
            return response_body
        except Exception as e:
            logger.exception("Error raised by inference endpoint")
            raise e

def embedding_func(text: str, input_type: str = "GENERIC_RETRIEVAL") -> List[float]:
    """Call out to Bedrock embedding endpoint with a single text."""
    # replace newlines, which can negatively affect performance.
    text = text.replace(os.linesep, " ")

    # Create embedding
    request_body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": input_type,
            "embeddingDimension": embedding_dim,
            "text": {"truncationMode": "END", "value": text},
        },
    }

    response = invoke_model(input_body=request_body)
    embedding = response["embeddings"][0]["embedding"]
    return embedding

Create MPAA ratings from the synthetic dataset to generate children friendly ratings based on the genres.

In [ ]:
for title_data in data_dict:
    has_family = True if "family" in title_data["genres"] and "action" not in title_data["genres"] and "scifi" not in title_data["genres"] else False
    if has_family:
        title_data["mpaa_ratings"].append("ty-y")


Iterate each title and generate embedding by invoking the embedding helper function.

In [ ]:
import time
for idx, title in enumerate(data_dict):
    if idx > 0 and idx % 1000 == 0:
        time.sleep(60)
    title["embedding"] = embedding_func(title["title_text"])

## Ingest catalog data into Amazon Opensearch Collection

In [ ]:
from oss_utils import OpenSearchHelper

oss_host = "provide your opensearch serverless host" # OpenSearch serverless hostname without the https. You can find the Opensearch Serverless endpoint from the CDK output: OpenSearchServerlessStack.OpenSearchCollectionEndpoint without the https://
oss_helper = OpenSearchHelper(host=oss_host)
index_name = "title_catalog_index" # The index name for ingesting the title catalog dataset. You can find the index name from the CDK output: OpenSearchServerlessStack.OpenSearchCollectionTitleCatalogIndex

Generate batches to speed up the data ingestion process

In [ ]:
from opensearchpy.helpers import bulk

data_bulk_collection = []
for index_data in data_dict:
    data_bulk = {}
    data_bulk["_index"] = index_name
    data_bulk["_source"] = index_data
    data_bulk_collection.append(data_bulk)

In [ ]:
from itertools import batched

batch_size = 20
oss_bulk_batches = [list(batch) for batch in batched(data_bulk_collection, batch_size)]

In [ ]:
for oss_bulk_batch in oss_bulk_batches:
    results = oss_helper.bulk_index(oss_bulk_batch)

# Process Open Source Movie Data
In addition to ingesting the synthetic movie catalog data, we also curated the movie catalog for an open source movie called [Tears of Steel](https://mango.blender.org/download/). In the following section, we'll ingest the title information for this movie into OpenSearch Serverless collection so that it could be a part of the movie catalog available for search and retrieval. 

In [ ]:
with open("../../data/exclusive_movie_dataset.json", "r") as f:
    data = f.read()

In [ ]:
exclusive_data_dict = json.loads(data)

In [ ]:
import time
for idx, title in enumerate(exclusive_data_dict):
    if idx > 0 and idx % 1000 == 0:
        time.sleep(60)
    title["embedding"] = embedding_func(title["title_text"])

In [ ]:
from opensearchpy.helpers import bulk

data_bulk_collection = []
for index_data in exclusive_data_dict:
    data_bulk = {}
    data_bulk["_index"] = index_name
    data_bulk["_source"] = index_data
    data_bulk_collection.append(data_bulk)

In [ ]:
from itertools import batched

batch_size = 20
oss_bulk_batches = [list(batch) for batch in batched(data_bulk_collection, batch_size)]

In [ ]:
for oss_bulk_batch in oss_bulk_batches:
    results = oss_helper.bulk_index(oss_bulk_batch)

# Ingest Catalog Title Posters to S3 Bucket
In addition to the synthetic movie catalog data, we also synthesized the posters images relevant for each movie. These posters would be shown in the web application when the matching recommended movies are presented. In the following section, we'll upload the images to an S3 bucket.

In [ ]:
media_content_bucket = "Your media content bucket" #use the bucket name created by CDK stack with output: NovaSonicSolutionBackendStack.MediaContentBucketName 
!cd ../../data/movie_posters && unzip images.zip 
!aws s3 cp --recursive ../../data/movie_posters/images/ s3://{media_content_bucket}/posters/

## Insert Featured movie into DynamoDB table
The following section inserts the movie URL to a dynamoDB table so that it could be referenced as a playback URL when the user click on the movie.

In [ ]:
def insert_item(title_id, **attributes):
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table('title_video_mapping') # or the output from CDK stack: NovaSonicSolutionBackendStack.DynamoDbTablesTitleVideoMappingTableNameE1E2504B  
    
    item = {'title_id': title_id, **attributes}
    table.put_item(Item=item)

In [ ]:
insert_item("tt2285752", video_name='tears-of-steel.mp4', video_s3_url=f"s3://{media_content_bucket}/input/bda/tears-of-steel.mp4")